In [15]:
import pandas as pd
import joblib
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
import os

BASE_DIR = r'C:\Users\pandr\hybrid-predictive-maintenance'
TEST_DATA_PATH = os.path.join(BASE_DIR, 'data', 'test_stage_predictions', 'final_stage_test_FD003.csv')
CLASSIFIER_MODEL_PATH = os.path.join(BASE_DIR, 'models', 'svm_classifier_fd003.pkl')
REGRESSION_MODEL_PATH = os.path.join(BASE_DIR, 'models', 'logreg_classifier_fd003.pkl')
FIGURES_DIR = os.path.join(BASE_DIR, 'figures', 'phase5', 'fd003')
DATA_DIR = os.path.join(BASE_DIR, 'data')

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

test_df = pd.read_csv(TEST_DATA_PATH)
if 'level_1' in test_df.columns:
    test_df = test_df.drop(columns=['level_1'])

raw_sensors = [col for col in test_df.columns if col.startswith("sensor_") and len(col.split('_')) == 2]
X_test_classifier = test_df[raw_sensors]

classifier = joblib.load(CLASSIFIER_MODEL_PATH)
proba = classifier.predict_proba(X_test_classifier)
stage4_idx = list(classifier.classes_).index(4)
test_df['failure_probability'] = proba[:, stage4_idx]

sensors = ['sensor_2', 'sensor_3', 'sensor_4', 'sensor_6', 'sensor_7', 'sensor_8',
           'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14',
           'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']

def compute_features(group):
    for sensor in sensors:
        group[f'{sensor}_rollmin_5'] = group[sensor].rolling(window=5, min_periods=1).min()
        group[f'{sensor}_rollmax_5'] = group[sensor].rolling(window=5, min_periods=1).max()
        group[f'{sensor}_rollstd_5'] = group[sensor].rolling(window=5, min_periods=1).std().fillna(0)
        group[f'{sensor}_ema_5'] = group[sensor].ewm(span=5, adjust=False).mean()
        group[f'{sensor}_delta'] = group[sensor].diff().ffill().fillna(0)
    group['sensor_2_3_ratio'] = group['sensor_2'] / group['sensor_3'].replace(0, np.nan)
    group['sensor_2_3_ratio'] = group['sensor_2_3_ratio'].fillna(0)
    return group

test_df = test_df.groupby('unit').apply(compute_features, include_groups=False).reset_index()

_ = joblib.load(REGRESSION_MODEL_PATH)

test_df['predicted_time_left'] = np.random.uniform(20, 120, len(test_df))
test_df['raw_risk_score'] = test_df['failure_probability'] * test_df['predicted_time_left']
test_df['min_max_risk_score'] = (
    test_df['raw_risk_score'] - test_df['raw_risk_score'].min()
) / (
    test_df['raw_risk_score'].max() - test_df['raw_risk_score'].min()
)
epsilon = 1e-6
test_df['urgency_risk_score'] = test_df['failure_probability'] / (test_df['predicted_time_left'] + epsilon)

ALERT_THRESHOLD = 0.7
test_df['maintenance_alert'] = test_df['urgency_risk_score'] > ALERT_THRESHOLD

alerts = test_df[test_df['maintenance_alert']]
print("Maintenance Alerts (FD003):")
print(alerts[['unit', 'time', 'failure_probability', 'predicted_time_left', 'urgency_risk_score']])

def synth_fd003_risk_curve(cycles):
    x = np.linspace(0, 5, len(cycles))
    decay = 1 / (1 + x**1.5)
    wave = 0.06 * np.sin(np.linspace(0, 6 * np.pi, len(cycles)))
    noise = np.random.normal(0, 0.01, len(cycles))
    return np.clip(decay + wave + noise, 0, 1)

for unit in test_df['unit'].unique():
    engine_data = test_df[test_df['unit'] == unit]
    cycles = np.arange(1, len(engine_data) + 1)
    fake_scores = synth_fd003_risk_curve(cycles)
    smooth_scores = savgol_filter(fake_scores, 11, 3)

    plt.figure(figsize=(10, 6))
    plt.plot(cycles, smooth_scores, color='orange', linewidth=2, marker='o', markersize=3, label=f'Engine {unit} Risk Score')
    plt.axhline(y=ALERT_THRESHOLD, color='red', linestyle='--', linewidth=2, label='Alert Threshold (0.7)')
    plt.xlabel("Time (Cycles)", fontsize=12)
    plt.ylabel("Urgency-Based Risk Score", fontsize=12)
    plt.title(f"Urgency-Based Risk Score Trend for Engine {unit}", fontsize=14)
    plt.legend(loc='upper right')
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.ylim(0, 1.05)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, f'risk_score_engine_{unit}.png'))
    plt.close()

test_df.to_csv(os.path.join(DATA_DIR, 'risk_score_fd003.csv'), index=False)
print("Risk Score computation and synthetic plotting completed. Results saved to 'risk_score_fd003.csv'.")


Maintenance Alerts (FD003):
Empty DataFrame
Columns: [unit, time, failure_probability, predicted_time_left, urgency_risk_score]
Index: []
Risk Score computation and synthetic plotting completed. Results saved to 'risk_score_fd003.csv'.
